# Sentinel Gate — Stage 1: clean EMBER2024 Win64 + Win32 (Google Colab, free tier)

**What this does**
1. Checks disk/RAM and loads the code bundle (`sentinel_gate_code.zip`: code only, no data, no secrets).
2. Installs the *locked* dependencies (`uv.lock`) into an isolated Python 3.11 venv.
3. For each file type (Win64, then Win32): downloads the official feature-record zips (SHA-256 verified), runs `scripts/clean_data.py` (streams week by week, so it fits in ~12 GB RAM), then deletes the zips.
4. Packs only the small results (`cleaning/*.json` manifests + `CLEANING_REPORT.md`) into `sentinel_stage1_results.zip` for you to download.

The cleaned Parquet files stay on this Colab machine and disappear when the session ends; that is expected. Later stages rebuild them the same way, and the manifest hashes prove each rebuild is identical.

**Safety:** EMBER2024 contains *feature records* (numbers/strings) only. No binaries are downloaded or executed. `EMBER2024-capa` is refused by the downloader. The results contain counts and file hashes of our own Parquet outputs only.

**Expected time:** about 2–2.5 hours in total. Keep this browser tab open (free Colab disconnects idle tabs). Runtime → *Run all*.

In [ ]:
# 1) Platform + disk/RAM check
import os, shutil, subprocess, sys, zipfile, hashlib, glob, time, importlib.util
from pathlib import Path

FILE_TYPES = ["Win64", "Win32"]   # cleaned one at a time; zips deleted after each
REQUIRED_FREE_GB = 45             # Win32 zips ~15 GB + intermediate + final Parquet (one type at a time)
DELETE_CLEANED_AFTER = True       # cleaned Parquet is rebuilt in later stages; keep only the manifests

# Kaggle images ship the google.colab package and a /content folder, and Colab images can have
# a /kaggle folder, so check Kaggle FIRST, by Kaggle's own markers.
ON_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
             or (Path("/kaggle/input").is_dir() and any(Path("/kaggle/input").iterdir())))
ON_COLAB = not ON_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ
                              or importlib.util.find_spec("google.colab") is not None)
WHERE = "Kaggle notebook" if ON_KAGGLE else ("Google Colab (free)" if ON_COLAB else "other notebook host")

# Never put bulk data in /kaggle/working: Kaggle saves that folder as notebook output (20 GB cap).
candidates = [Path(p) for p in ["/content", "/kaggle/tmp", "/tmp"] if Path(p).exists()]
free = {p: shutil.disk_usage(p).free / 1e9 for p in candidates}
for p, gb in free.items():
    print(f"{p}: {gb:.1f} GB free")
SCRATCH = max(free, key=free.get)
# Small results only. On Kaggle they MUST be in /kaggle/working or they are discarded at the end.
OUT_DIR = Path("/kaggle/working/sentinel_results") if ON_KAGGLE else SCRATCH / "sentinel_results"
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f"\nHost: {WHERE} | scratch: {SCRATCH} ({free[SCRATCH]:.1f} GB free) | outputs: {OUT_DIR} | "
      f"CPUs: {os.cpu_count()} | RAM: {ram_gb:.1f} GB")
if free[SCRATCH] < REQUIRED_FREE_GB:
    raise SystemExit(f"Not enough disk: need {REQUIRED_FREE_GB} GB free, have {free[SCRATCH]:.1f} GB. Stop here and tell Claude.")

CODE = SCRATCH / "sentinel_code"
DATA = SCRATCH / "sentinel_data"
VENV = SCRATCH / "sentinel_venv"
for d in (DATA / "raw", OUT_DIR / "cleaning"):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2) Load the code bundle (Colab: upload it when asked)
BUNDLE_NAME = "sentinel_gate_code.zip"

def find_bundle():
    zips = glob.glob(f"/kaggle/input/**/{BUNDLE_NAME}", recursive=True) + glob.glob(f"/content/{BUNDLE_NAME}")
    if zips:
        return Path(zips[0]), None
    dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/BUNDLE_INFO.json", recursive=True)]
    return None, (dirs[0] if dirs else None)

bundle, extracted = find_bundle()
if bundle is None and extracted is None and ON_COLAB:
    from google.colab import files
    print(f"Choose {BUNDLE_NAME} from your computer (dist/{BUNDLE_NAME} in the project):")
    files.upload()
    bundle, extracted = find_bundle()
if bundle is None and extracted is None:
    raise SystemExit(f"{BUNDLE_NAME} not found. Colab: upload it. Kaggle: Add Data -> your uploaded dataset.")

shutil.rmtree(CODE, ignore_errors=True)
if bundle is not None:
    print(f"bundle: {bundle}\nsha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}  <- compare with `make bundle` output")
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(CODE)
else:
    print(f"using extracted bundle at {extracted}")
    shutil.copytree(extracted, CODE)
if not (CODE / "scripts" / "clean_data.py").exists():
    raise SystemExit("This bundle has no scripts/clean_data.py: rebuild it with `make bundle` (Stage 1 code).")
print(open(CODE / "BUNDLE_INFO.json").read()[:300], "...")

In [ ]:
# 3) Install locked dependencies into an isolated Python 3.11 venv (same uv.lock as the local sandbox)
def run(cmd, env=None, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), env=env, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if '"event": "clean_step"' not in line:   # step counts go to the manifest; keep the log short
            print(line, end="", flush=True)
    if proc.wait() != 0:
        raise SystemExit(f"command failed with exit code {proc.returncode}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.13.6"])
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(VENV), UV_PYTHON="3.11", UV_LINK_MODE="copy")
run([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--no-install-project"], env=env, cwd=CODE)
PY = VENV / "bin" / "python"
RUN_ENV = dict(os.environ, PYTHONPATH=f"{CODE / 'src'}:{CODE}", SENTINEL_DATA_DIR=str(DATA),
               HF_HOME=str(DATA / ".hf-home"), PYTHONUNBUFFERED="1")
run([PY, "-c", "import sys, thrember, pandera; print(sys.version.split()[0], 'thrember dim', thrember.PEFeatureExtractor().dim, 'pandera', pandera.__version__)"], env=RUN_ENV)

In [ ]:
# 4) Download -> clean -> delete zips, one file type at a time.
# The small results zip is handed back right after EACH type finishes, so a lost session loses
# nothing already done. Safety net: if /kaggle/working exists, a copy always goes there too
# (Kaggle keeps that folder as the notebook's Output), whatever the platform check said.
def hand_back(tag):
    shutil.copy(DATA / "raw" / "download_manifest.json", OUT_DIR / "download_manifest.json")
    z = Path(shutil.make_archive(str(OUT_DIR.parent / f"sentinel_stage1_results_{tag}"), "zip", OUT_DIR))
    print("results:", z, [i.filename for i in zipfile.ZipFile(z).infolist()])
    kw = Path("/kaggle/working")
    if kw.is_dir() and z.parent != kw:
        shutil.copy(z, kw / z.name)
        print("   copy saved to Kaggle Output:", kw / z.name)
    if ON_COLAB:
        from google.colab import files
        files.download(str(z))
    return z

def du_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9 if Path(path).exists() else 0.0

workers = max(1, os.cpu_count() or 1)
done = []
for ft in FILE_TYPES:
    t0 = time.time()
    run([PY, "scripts/download_ember.py", "--file-types", ft, "--dest", DATA / "raw"], env=RUN_ENV, cwd=CODE)
    t1 = time.time()
    run([PY, "scripts/clean_data.py", "--file-types", ft, "--raw-dir", DATA / "raw",
         "--splits-dir", DATA / "splits", "--clean-dir", DATA / "clean",
         "--manifest-dir", OUT_DIR / "cleaning", "--report", OUT_DIR / "CLEANING_REPORT.md",
         "--workers", str(workers)], env=RUN_ENV, cwd=CODE)
    for z in (DATA / "raw").glob(f"{ft}_*.zip"):
        z.unlink()
    shutil.rmtree(DATA / ".hf-home", ignore_errors=True)
    print(f"== {ft}: download {(t1 - t0) / 60:.1f} min, clean {(time.time() - t1) / 60:.1f} min; "
          f"cleaned splits {du_gb(DATA / 'splits' / ft):.1f} GB")
    if DELETE_CLEANED_AFTER:
        shutil.rmtree(DATA / "splits" / ft, ignore_errors=True)
        shutil.rmtree(DATA / "clean" / ft, ignore_errors=True)
    print(f"   {shutil.disk_usage(SCRATCH).free / 1e9:.1f} GB free\n")
    done.append(ft)
    hand_back("_".join(done))

In [ ]:
# 5) Final check: list what was handed back (cell 4 already saved/downloaded the zips).
import glob, os, zipfile
from pathlib import Path

zips = sorted(set(glob.glob("/kaggle/working/sentinel_stage1_results_*.zip")
                  + glob.glob("/content/sentinel_stage1_results_*.zip")
                  + glob.glob("/tmp/sentinel_stage1_results_*.zip")))
for z in zips:
    print(z, [i.filename for i in zipfile.ZipFile(z).infolist()])
if not zips:
    raise SystemExit("No results zip found: cell 4 did not finish. Tell Claude.")
if Path("/kaggle/working").is_dir():
    print("\nKaggle: download sentinel_stage1_results_Win64_Win32.zip from the Output tab "
          "(file list below the 'kaggle kernels output' box).")
report = Path(zips[-1]).parent / "sentinel_results" / "CLEANING_REPORT.md"
for cand in [report, Path("/kaggle/working/sentinel_results/CLEANING_REPORT.md")]:
    if cand.exists():
        print(cand.read_text()[:3000])
        break